# 5. Production Prediction

This notebook demonstrates how the trained machine learning model can be used to predict actual production quantity (`actual_qty`) for a new production order.

The saved model pipeline, `production_model.joblib`, contains the selected Gradient Boosting model and the preprocessing transformations required to prepare input data for prediction.

The prediction process follows four steps:

1. Load the saved model pipeline.
2. Prepare a new production order using the required input features.
3. Recreate the engineered features used during model training.
4. Generate and interpret the predicted actual production quantity.

The prediction is based on production characteristics and resource information supplied before production begins. This allows the model to estimate expected output before the production process is completed.

**Input:** `production_model.joblib` and a DataFrame describing the new production order.

**Output:** Predicted actual production quantity, expressed as a whole number of shoes.

This notebook represents the predictive component of the project. The resulting prediction can subsequently support prescriptive analytics by helping compare expected production output with the planned production target.

## 5.1 Import Required Libraries

Pandas is used to organise the information for a new production order into a DataFrame. Joblib is used to load the saved machine learning pipeline.

The saved pipeline includes both the preprocessing transformations and the trained regression model. Loading the complete pipeline ensures that the input data is processed consistently with the data used during model training.

Using the saved pipeline avoids the need to retrain the model whenever a new production prediction is required.

In [1]:
import pandas as pd
import joblib

## 5.2 Define the Production Prediction Function

The `predict_production()` function provides a reusable way to generate production forecasts for new input records.

The function performs three main operations.

### 5.2.1 Load the Saved Model

The saved pipeline is loaded from `production_model.joblib` using Joblib.

The pipeline contains the fitted preprocessing steps and the selected Gradient Boosting regression model. This allows new production records to pass through the same transformations used during model development.

### 5.2.2 Recreate the Engineered Features

Two engineered features are calculated from the supplied input variables:

- `machine_worker_ratio = machine_count / no_of_workers`
- `avg_hours_per_worker = working_hours / no_of_workers`

These features represent machines per worker and average recorded working hours per worker, respectively.

They must be calculated consistently with the feature engineering performed during model training. If the formulas or feature names differ, the prediction pipeline may receive inputs that do not match the structure expected by the trained model.

The function creates a copy of the input DataFrame before adding the features. This prevents the original input DataFrame from being modified by these operations.

The calculations require `no_of_workers` to be greater than zero. The supplied input should also contain valid values for the required columns.

### 5.2.3 Generate the Prediction

The prepared DataFrame is passed to the saved pipeline, which applies the fitted preprocessing transformations and generates a prediction for each input record.

The resulting predictions are rounded to whole numbers and converted to integers because the output represents a quantity of shoes.

The function can process multiple production orders in a single DataFrame. It returns one predicted actual production quantity for each row.

The output is an estimate of expected production, not a guarantee of the quantity that will ultimately be produced.

In [2]:
def predict_production(new_data, model_path='production_model.joblib'):
    """
    Loads the saved model and predicts actual_qty for new data.
    new_data: A pandas DataFrame containing the pre-production features.
    """
    # 1. Load the saved pipeline
    pipeline = joblib.load(model_path)

    # 2. Apply the exact same feature engineering
    # .copy() avoids modifying the original input dataframe
    df_pred = new_data.copy()
    df_pred['machine_worker_ratio'] = df_pred['machine_count'] / df_pred['no_of_workers']
    df_pred['avg_hours_per_worker'] = df_pred['working_hours'] / df_pred['no_of_workers']

    # 3. Predict
    predictions = pipeline.predict(df_pred)

    # Return whole numbers, since the output is a quantity of shoes
    return predictions.round().astype(int)

## 5.3 Prepare a New Production Order

A new production order is represented as a Pandas DataFrame containing the 14 input columns required by the prediction function.

These columns describe the production period, factory unit, product characteristics, workforce, working hours, available machines, order category, and known bottleneck conditions.

| Input column | Purpose |
|---|---|
| `financial_month` | Represents the financial month associated with the production order. |
| `week_no` | Indicates the week number used to describe the production period. |
| `week_of_month` | Represents the position of the week within the month. |
| `factory_unit` | Identifies the factory unit responsible for production. |
| `product_family` | Identifies the product family, such as Ladies or Gents. |
| `item_name` | Identifies the specific shoe item. |
| `size_id` | Specifies the shoe size. |
| `grade` | Represents the product grade. |
| `rate` | Provides the recorded rate value used by the model. |
| `no_of_workers` | Specifies the number of workers assigned. |
| `working_hours` | Provides the recorded working hours for the production order. |
| `machine_count` | Specifies the number of machines available. |
| `order_category` | Identifies the order category, such as School, Forces/Police, or Others. |
| `bottleneck_flag` | Indicates a known machine fault or constraint using 0 or 1. |

The values supplied should describe the production conditions expected for the order being forecast.

### 5.3.1 Why Are These Inputs Required?

The model learned relationships between the input features and `actual_qty` during training. Providing the same types of information allows the saved pipeline to apply those learned relationships to a new production scenario.

For example, changing the number of workers, working hours, or machine count changes the information supplied to the model and may therefore change the predicted output. The direction and size of the change must be determined from the model's prediction rather than assumed in advance.

The input should contain valid values that follow the formats expected by the trained pipeline. The engineered ratio columns do not need to be entered manually because the prediction function calculates them from the supplied resource variables.

### 5.3.2 Columns Not Required for Prediction

The following columns are not required by this prediction function:

- `planned_qty`
- `production_gap_qty`
- `total_duration_per_shoe_mins`
- `week_start_date`
- `week_end_date`

These columns are excluded from the model inputs used in this notebook. In particular, `production_gap_qty` depends on actual production quantity and would reveal information about the target if supplied as a predictor.

The date columns are also not passed into the model. The prediction therefore depends on the supplied production features rather than directly using the calendar dates.

### 5.3.3 Unseen Categories

A new production order may contain a category or item that was not present in the training data.

The one-hot encoder is configured to ignore unknown categories in the categorical features it handles. The target encoder's treatment of unseen items depends on its fitted configuration; under the intended fallback behaviour, an unseen item can receive an overall target-based estimate.

This allows prediction to be attempted for new categories, but it does not guarantee that the resulting estimate will be as reliable as a prediction for a well-represented category.

### 5.3.4 Importance of Input Validity

The prediction function assumes that the required columns exist and contain valid values.

In particular, `no_of_workers` must be greater than zero because it is used as the denominator in both engineered features. The `bottleneck_flag` should be supplied as 0 or 1 and should only indicate a condition that is already known when the prediction is made.

The model's usefulness depends on the supplied inputs accurately representing the conditions under which the production order will be carried out.

### 5.3.5 Example Production Order

A sample production order is created to demonstrate how the prediction function can be used.

The example specifies the financial month, week, factory unit, product family, item, size, grade, rate, workforce size, working hours, machine count, order category, and bottleneck status.

These values provide a hypothetical production scenario for testing the prediction workflow. They should not be interpreted as a verified production plan or as a statement about the actual resources assigned to a real order.

The DataFrame is displayed so that the supplied values can be checked before generating the prediction.

For a different scenario, the input values can be changed and the prediction function can be executed again. This makes it possible to examine how alternative resource configurations are reflected in the model's forecasts.

In [7]:
new_order = pd.DataFrame([{
    'financial_month': 'Apr',
    'week_no': 4,
    'week_of_month': 4,
    'factory_unit': 'Factory Unit 01',
    'product_family': 'Ladies',
    'item_name': 'BINOLI (VAST) - BROWN',
    'size_id': 10,
    'grade': 'A',
    'rate': 4500.00,
    'no_of_workers': 18,
    'working_hours': 800.5,
    'machine_count': 5,
    'order_category': 'Others',
    'bottleneck_flag': 0
}])

new_order

,financial_month,week_no,week_of_month,factory_unit,product_family,item_name,size_id,grade,rate,no_of_workers,working_hours,machine_count,order_category,bottleneck_flag
0,Apr,4,4,Factory Unit 01,Ladies,BINOLI (VAST) - BROWN,10,A,4500.0,18,800.5,5,Others,0


## 5.4 Generate the Production Prediction

The prepared production order is passed to `predict_production()`.

The function loads the saved pipeline, calculates the two engineered resource features, applies the fitted preprocessing transformations, and generates a prediction for `actual_qty`.

The predicted value is returned as a whole number because the quantity of shoes is expressed as an integer.

The output represents the model's estimate of the actual production quantity under the supplied conditions. It is a predictive estimate based on patterns learned from historical records rather than a direct measurement of future production.

The prediction can be used as an input to the next stage of the project, where expected production output can be compared with a planned quantity to support production planning.

In [8]:
predicted_qty = predict_production(new_order)

print(f"Predicted Actual Production Qty: {predicted_qty[0]} shoes")

Predicted Actual Production Qty: 48 shoes


## 5.5 Interpret the Prediction

The printed output shows the model's predicted actual production quantity for the supplied production order.

For example, if the model returns a prediction of 40, this means that the model estimates an output of approximately 40 shoes under the conditions represented by the input record.

The result should be interpreted alongside the evaluation metrics obtained during model selection:

- **Test MAE = 4.577:** The average absolute prediction error was approximately 4.58 production units on the held-out test records.
- **Test RMSE = 6.000:** Larger prediction errors receive greater weight, and the test RMSE was approximately 6 production units.
- **Test R² = 0.791:** The model explained approximately 79.1% of the variation in the held-out actual production quantities relative to the test-set mean baseline.

These metrics describe performance across the test set. They do not establish the exact error for this individual production order, and the actual quantity may differ from the prediction.

The predicted quantity should therefore be treated as a planning estimate rather than a guaranteed result.

### 5.5.1 Examine Alternative Production Scenarios

Different input scenarios can be created by changing variables such as `no_of_workers`, `working_hours`, `machine_count`, or `bottleneck_flag`.

The prediction function can then be rerun to compare the model's estimated output under those scenarios.

Such comparisons can help investigate how the model responds to different combinations of resources and operating conditions. However, a change in a prediction does not prove that changing a single resource will cause the same change in real production.

For meaningful comparisons, keep the other input values constant when examining one feature at a time, and ensure that the scenarios are operationally realistic.

The model's predictions can support decision-making, but final resource allocation should also consider production constraints and relevant operational knowledge.
